# Machine Translation Project: Russian to English (Biomedical Domain)

This Jupyter Notebook serves as an **organized workflow** for training and evaluating a **Neural Machine Translation (NMT) model** using **Sequence-to-Sequence with Attention**.

In [ ]:
# Clone repo from github
#!git clone https://fboglind@github.com/fboglind/lab4_mt_project.git

In [ ]:
# Paperspace Gradient specific code
import os

# Set base directory
base_dir = "/notebooks/lab4_mt_project"
os.chdir(base_dir)

# Verify that the directory change was successful
print("Current working directory:", os.getcwd())

In [ ]:
# Install dependencies (if not installed)
%pip install torch pandas matplotlib seaborn sentencepiece sacrebleu

## **1. Create parallel corpus and test set**
- Reads files containing abstracts
- Creates a parallel corpus
- Creates a test set

The data for this project consists of parallel abstracts which can be retrieved from Medline using a script: wmtbio22_train_data.py.
Biopython and a valid email is needed to access the data in Medline.
https://github.com/biomedical-translation-corpora/corpora/blob/master/wmtbio22_train_data.py

In [ ]:
# Assuming that the Medline data has been downloaded and extracted to abstracts/ we create parallel corpus and test set
%mkdir -p data
!python3 scripts/preprocess_wmt22.py

## **2. Load & Explore Data**


- Load the parallel corpus
- Analyze text length distributions
- Check for alignment issues
- Clean the data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
df = pd.read_csv("data/parallel_corpus.tsv", sep="\t", encoding="utf-8")

# Basic statistics
print(df.describe())
print(df.head())

# Plot text length distributions
df["Russian_Length"] = df["Russian"].apply(lambda x: len(x.split()))
df["English_Length"] = df["English"].apply(lambda x: len(x.split()))

sns.histplot(df["Russian_Length"], bins=30, kde=True, label="Russian", color="blue")
sns.histplot(df["English_Length"], bins=30, kde=True, label="English", color="orange", alpha=0.6)
plt.legend()
plt.title("Distribution of Abstract Lengths (Words)")
plt.xlabel("Number of Words")
plt.ylabel("Count")
plt.show()

Let's remove some of the 'misaligned' abstracts, while also perform some basic cleaning

In [ ]:
!python3 scripts/clean_parallel_corpus.py --input data/parallel_corpus.tsv --output data/cleaned_parallel_corpus_initial.tsv

In [ ]:
# Check the cleaned data
df = pd.read_csv("data/cleaned_parallel_corpus_initial.tsv", sep="\t", encoding="utf-8")
# Display one column per row for the first lines of the dataframe
for index, row in df.head(10).iterrows():
    print(f"Row {index}:")
    print(f"Russian: {row['Russian']}")
    print(f"English: {row['English']}")
    print(f"Russian_Length: {row['Russian_Length']}")
    print(f"English_Length: {row['English_Length']}")
    print("-" * 50)

Just by looking at the first lines we can see that the Russian text contain some reocurring opening phrases that have no counterpart in the English text. 

We scan the text for such common opening phrases while also checking if they have any corresponding English phrases:

In [ ]:
!python3 scripts/extract_openers_from_english.py

Let's remove these opening phrases from the Russian texts together with common headers (ex. "Результаты и обсуждение" - "Results and discussion" etc.) as these are not present in the English text either.

In [ ]:
!python3 scripts/clean_russian_openers.py
!head data/cleaned_parallel_corpus.tsv

The abstracts are quite long, we therefore split them into sentences before further tokenization.

In [ ]:
!python3 scripts/split_abstracts.py --input data/cleaned_parallel_corpus.tsv --output data/sentence_aligned_corpus.tsv

In [ ]:
# We split the test into sentences as well
!python3 scripts/split_single_column.py \
  --input data/test_raw_ru.txt \
  --output data/test_sentence_set.txt

In [ ]:
# Check for misaligned sentences
import pandas as pd
df = pd.read_csv("data/sentence_aligned_corpus.tsv", sep="\t", encoding="utf-8")

# Check for misaligned rows
misaligned = df[(df["Russian"].str.len() < 10) | (df["English"].str.len() < 10)]
print(f"⚠️ Found {len(misaligned)} potentially broken rows.")

# Check for newline artifacts
newline_issues = df[df["Russian"].str.contains("\n") | df["English"].str.contains("\n")]
print(f" Found {len(newline_issues)} rows with embedded newlines.")


Let's scan the Russian text for Latin terms

In [ ]:
import re

def contains_latin(text):
    return bool(re.search(r"[A-Za-z]{2,}", text))

# Filter or flag those rows
df_with_latin = df[df["Russian"].apply(contains_latin)]
print(f"Found {len(df_with_latin)} rows with Latin characters in Russian.")

for i, row in df.iterrows():
    if contains_latin(row["Russian"]):
        print(f"[{i}] {row['Russian']}")


In [ ]:
# Extract numbers and Latin characters and save them to a file
!python3 scripts/extract_numbers_and_latin.py

## **3. Preprocessing: SentencePiece Tokenization**

- Extract source language data
- Train a SentencePiece model
- Tokenize the training and test sets

In [ ]:


# Train SentencePiece. Output: spm_ru_en.model, spm_ru_en.vocab
!mkdir -p models
!python3 scripts/sentencepiece_train.py

In [ ]:
!head -n20 models/spm_ru_en.vocab

In [ ]:
# Apply SentencePiece Tokenization
!python3 scripts/apply_sentencepiece.py --input data/sentence_aligned_corpus.tsv --output data/spm_parallel_corpus.tsv --model models/spm_ru_en.model --to_parallel

In [ ]:
!head -n 5 data/spm_parallel_corpus.tsv

We must also tokenize the test set

In [ ]:
# Apply sentencepiece to the test set
!python3 scripts/apply_sentencepiece.py \
  --input data/test_sentence_set.txt \
  --output data/preprocessed_test_set.tsv\
  --is_test_set

In [ ]:
!head -n 5 data/preprocessed_test_set.tsv

## **4. Model Training**

- Train the sequence-to-sequence model
- Use label smoothing to improve training stability
- Save the trained model

In [ ]:
# Train the model with accumulated gradients (batch size 8, gradient accumulation steps 4)
!python3 scripts/seq2seq_train.py \ 
  --batch-size 8 \
  --accum-steps 4

# Or to run in background use:
# ./run_training.sh

## **5. Translation & Evaluation**

- Extract reference translations
- Translate test data
- Evaluate performance using BLEU and chrF

In [ ]:
# Translate test set
!python3 scripts/translate_wmt_test.py

In [ ]:
# Evaluate translations
!python3 scripts/evaluate_translations.py \
  --hyp data/test_hypothesis_en.txt \
  --ref data/test_reference_en.txt \ 
  


## **6. Next Steps & Improvements**

- Compare GRU vs. LSTM architectures
- Improve handling of long sequences (segmenting abstracts)
- Fine-tune hyperparameters for better results